# GeoSR-4 — DINOv2 perceptual-loss backbone ablation (Colab GPU)

D023's perceptual loss uses VGG16, ImageNet-pretrained on natural photos -- a real domain mismatch for Sentinel-2 satellite imagery. Research (D042) found DINOv3 has a checkpoint pretrained on SAT-493M (satellite imagery) that would be a much better domain match, but it's gated (Meta license, manual approval -- request already submitted, pending).

This notebook tests **DINOv2** (`facebook/dinov2-small`, free, no gating) as an interim comparison -- isolates whether a stronger backbone helps *on its own architectural merits*, before the domain-match question can even be tested. Config matches D028/D029's VGG "quality run" **exactly** (30 epochs, batch 8, lr 1e-4, lambda-perceptual 0.01, ICNR on) -- only `--perceptual-backbone` changes, so this is a clean, isolated comparison against those already-logged numbers.

**Before running:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies
`transformers` is new here -- needed to load the DINOv2 backbone.

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image torchvision transformers

## 2. Download the dataset (cross-sensor split only, ~2.1 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Train: DINOv2 perceptual backbone (same protocol as D028/D029's VGG quality run)
Only `--perceptual-backbone dino` changed vs the original quality run -- epochs, batch size, lr, lambda-perceptual, ICNR (default on) all identical, so the eval numbers below are directly comparable to D028/D029.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # D025: fragmentation fix

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python ml/training/train_swinir.py \
  --epochs 30 \
  --batch-size 8 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --perceptual-backbone dino \
  --amp \
  --checkpoint-dir experiments/swinir_dino_quality \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_dino_quality/swinir_epoch29.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 4. (Later, once DINOv3 gated access is approved) Switch to the satellite-pretrained backbone
Don't run this yet -- the DINOv3 sat493m checkpoint request (D042) is still pending approval. Once the approval email arrives: log in with an HF token that has access, then rerun training with `--dino-model-id facebook/dinov3-vitl16-pretrain-sat493m`. That checkpoint is ViT-L (300M params) vs DINOv2-small (21M) -- much bigger, so `--batch-size` is dropped to 4 as a conservative starting guess against T4's 15GB (untested -- adjust down further if it OOMs).

In [ ]:
# Run this only after DINOv3 gated access is approved (D042):
# from huggingface_hub import login
# login()  # paste your HF token (must have access to facebook/dinov3-vitl16-pretrain-sat493m)
#
# !python ml/training/train_swinir.py \
#   --epochs 30 --batch-size 4 --embed-dim 60 --depths 2,2,2,2 \
#   --num-heads 6 --window-size 11 --lr 1e-4 \
#   --lambda-perceptual 0.01 --perceptual-backbone dino \
#   --dino-model-id facebook/dinov3-vitl16-pretrain-sat493m \
#   --amp --checkpoint-dir experiments/swinir_dino3_sat_quality --log-every 20
#
# !python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_dino3_sat_quality/swinir_epoch29.pt \
#   --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 5. Download the checkpoint

In [ ]:
from google.colab import files
files.download('experiments/swinir_dino_quality/swinir_epoch29.pt')